# Ensembles — Demo

Walk through Random Forest and Gradient Boosting end-to-end on self-contained sklearn datasets.

**Pipeline:**
1. Baseline — single decision tree
2. Random Forest — bagging in action
3. OOB error and feature importance
4. Gradient Boosting — sequential learning
5. Effect of learning rate and n_estimators
6. Side-by-side comparison
7. Regression with both

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

np.random.seed(42)

## 1 — Load Data and Baseline

Breast cancer: 30 numerical features, binary target.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Shape:', X.shape, '| Train:', X_train.shape[0], '| Test:', X_test.shape[0])

In [ ]:
# Single Decision Tree as baseline
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

print(f'Single tree -- Train acc: {tree.score(X_train, y_train):.3f}  Test acc: {tree.score(X_test, y_test):.3f}')
print(f'              Depth: {tree.get_depth()}  Leaves: {tree.get_n_leaves()}')

Single tree overfits — train accuracy near 1.0, but test accuracy is lower.

## 2 — Random Forest (Bagging)

Same data, but now 200 trees averaged together.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

print(f'Random Forest -- Train acc: {rf.score(X_train, y_train):.3f}  Test acc: {rf.score(X_test, y_test):.3f}')
print(f'                 OOB score: {rf.oob_score_:.3f}')

**OOB score** is close to the test accuracy — it's a free internal validation, no separate split needed.

## 3 — Effect of Number of Trees

In [ ]:
rows = []
for n in [1, 5, 10, 50, 100, 200, 500]:
    m = RandomForestClassifier(n_estimators=n, max_features='sqrt', n_jobs=-1, random_state=42)
    m.fit(X_train, y_train)
    rows.append({
        'n_trees':   n,
        'train_acc': round(m.score(X_train, y_train), 3),
        'test_acc':  round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(rows))

More trees → diminishing returns. Beyond ~100, you usually see only marginal gains.

## 4 — Feature Importance from Random Forest

In [ ]:
importance = pd.DataFrame({
    'feature':    feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(10).round(3))

In [ ]:
top = importance.head(10)
plt.figure(figsize=(10, 5))
plt.barh(top['feature'][::-1], top['importance'][::-1])
plt.xlabel('Importance')
plt.title('Top 10 Features — Random Forest')
plt.tight_layout()
plt.show()

## 5 — Gradient Boosting (Boosting)

Same data, but trees built sequentially — each one fixes previous errors.

In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,                  # SHALLOW trees - boosting prefers weak learners
    random_state=42
)
gbm.fit(X_train, y_train)

print(f'Gradient Boosting -- Train acc: {gbm.score(X_train, y_train):.3f}  Test acc: {gbm.score(X_test, y_test):.3f}')

## 6 — Effect of Learning Rate

Lower learning rate → cautious learning, often better generalisation (but needs more trees).

In [ ]:
rows = []
for lr in [0.01, 0.05, 0.1, 0.5, 1.0]:
    m = GradientBoostingClassifier(
        n_estimators=200, learning_rate=lr, max_depth=3, random_state=42
    ).fit(X_train, y_train)
    rows.append({
        'learning_rate': lr,
        'train_acc':     round(m.score(X_train, y_train), 3),
        'test_acc':      round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(rows))

Note how `learning_rate=1.0` likely overfits (high train, lower test). Smaller values are usually safer.

## 7 — Train vs Test Curve as Trees Are Added

The classic boosting plot — train error keeps falling, test error eventually rises.

In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.1, max_depth=3, random_state=42
).fit(X_train, y_train)

# staged_predict gives prediction at each boosting iteration
train_errors, test_errors = [], []
for y_train_pred in gbm.staged_predict(X_train):
    train_errors.append(1 - accuracy_score(y_train, y_train_pred))
for y_test_pred in gbm.staged_predict(X_test):
    test_errors.append(1 - accuracy_score(y_test, y_test_pred))

plt.figure(figsize=(10, 5))
plt.plot(train_errors, label='Train error')
plt.plot(test_errors, label='Test error')
plt.xlabel('Number of trees')
plt.ylabel('Error rate')
plt.title('Gradient Boosting — train vs test error per iteration')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8 — Side-by-Side Comparison

In [ ]:
models = {
    'Single Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=200, max_features='sqrt', n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    results.append({
        'Model':     name,
        'Train acc': round(m.score(X_train, y_train), 3),
        'Test acc':  round(m.score(X_test,  y_test),  3)
    })
print(pd.DataFrame(results))

## 9 — Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators':  [100, 200, 500],
    'max_depth':     [None, 10, 20],
    'max_features':  ['sqrt', 'log2']
}

grid = GridSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV accuracy: {grid.best_score_:.3f}')
print(f'Test accuracy:    {grid.score(X_test, y_test):.3f}')

## 10 — Regression Example

Same comparison on a regression problem — diabetes disease progression.

In [ ]:
data_r = load_diabetes()
X_r, y_r = data_r.data, data_r.target

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42
)

In [ ]:
models_r = {
    'Random Forest':     RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
}

results = []
for name, m in models_r.items():
    m.fit(X_train_r, y_train_r)
    pred = m.predict(X_test_r)
    results.append({
        'Model':    name,
        'Test R2':  round(r2_score(y_test_r, pred), 3),
        'Test MSE': round(mean_squared_error(y_test_r, pred), 1)
    })
print(pd.DataFrame(results))

## Summary

- **Single tree** overfits → train ≈ 1.0, test lower
- **Random Forest** averages many trees → reduces variance, OOB error for free
- **Gradient Boosting** sequentially corrects errors → often higher test accuracy
- Both produce **feature importance** that's more stable than a single tree
- **Learning rate** is the single most important GBM hyperparameter
- **GridSearchCV** standard way to tune both ensemble types
- Same API for classification and regression (`Classifier` vs `Regressor`)

> Random Forest = robust baseline. Gradient Boosting = highest accuracy with tuning.